# Notebook 2 — Feature Engineering

This notebook covers:
- Loading Parquet data
- Custom Transformer (`LogGoalTransformer`) — domain-specific feature
- String indexing, vector assembly
- Data lineage and error handling
- RDD vs DataFrame justification

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, log1p, count, when, isnull
from pyspark.ml import Transformer, Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.param.shared import HasInputCol, HasOutputCol
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable

import os
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Kickstarter - Feature Engineering") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready.")

Spark ready.


## 1. Load Parquet Dataset

In [2]:
PARQUET_PATH = "../data/parquet/kickstarter"

df = spark.read.parquet(PARQUET_PATH)
df = df.select("category", "country", "goal", "state",
               "launched_at", "deadline", "staff_pick")

print("Row count:", df.count())
df.printSchema()

Row count: 203341
root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- goal: double (nullable = true)
 |-- state: integer (nullable = true)
 |-- launched_at: integer (nullable = true)
 |-- deadline: integer (nullable = true)
 |-- staff_pick: boolean (nullable = true)



## 2. Custom Transformer — LogGoalTransformer

**Domain justification:** Campaign funding goals are highly right-skewed (some campaigns ask for millions, most ask for hundreds). Applying `log1p` normalises the distribution and prevents large goals from dominating distance-based and gradient algorithms.

In [4]:
class LogGoalTransformer(
    Transformer,
    HasInputCol,
    HasOutputCol,
    DefaultParamsReadable,
    DefaultParamsWritable
):
    """
    Custom MLlib Transformer that applies log1p to a numeric column.
    Designed for highly skewed funding goal distributions.
    Implements DefaultParamsReadable/Writable for MLlib serialization.
    """

    def __init__(self, inputCol="goal", outputCol="goal_log"):
        super(LogGoalTransformer, self).__init__()
        # _set() is the correct internal PySpark method to initialise Param values;
        # setInputCol/setOutputCol require the Params metaclass to have already
        # registered the param slots, which doesn't happen reliably via super().__init__()
        self._set(inputCol=inputCol, outputCol=outputCol)

    def _transform(self, dataset):
        return dataset.withColumn(
            self.getOutputCol(),
            log1p(col(self.getInputCol()))
        )

# Demonstrate standalone usage
log_tx = LogGoalTransformer(inputCol="goal", outputCol="goal_log")
df_log = log_tx.transform(df)

df_log.select("goal", "goal_log").describe().show()


+-------+------------------+--------------------+
|summary|              goal|            goal_log|
+-------+------------------+--------------------+
|  count|            203341|              203341|
|   mean|55736.491422880776|   8.451190355151567|
| stddev|1308667.4868899218|   1.811029085727608|
|    min|              0.01|0.009950330853168083|
|    max|             1.0E8|  18.420680753952364|
+-------+------------------+--------------------+



In [11]:
import numpy as np
import os
import csv

# matplotlib is broken in this environment (matplotlib.colors / .backends missing).
# Compute histogram buckets in Spark instead and save counts to CSV.
# The CSV can be opened in Excel or Tableau for visualisation if needed.

from pyspark.ml.feature import Bucketizer
from pyspark.sql.functions import count as spark_count

goal_splits     = list(np.linspace(0, 5_000_000, 61).tolist()) + [float("inf")]
goal_log_splits = list(np.linspace(0, 16, 61).tolist()) + [float("inf")]

bkt_goal = Bucketizer(splits=goal_splits,     inputCol="goal",     outputCol="goal_bucket",     handleInvalid="keep")
bkt_log  = Bucketizer(splits=goal_log_splits, inputCol="goal_log", outputCol="goal_log_bucket", handleInvalid="keep")

df_bucketed = bkt_log.transform(bkt_goal.transform(df_log))

goal_bins = (df_bucketed.groupBy("goal_bucket")
             .agg(spark_count("*").alias("n"))
             .orderBy("goal_bucket").collect())
log_bins  = (df_bucketed.groupBy("goal_log_bucket")
             .agg(spark_count("*").alias("n"))
             .orderBy("goal_log_bucket").collect())

os.makedirs("../output", exist_ok=True)

# Save to CSV for external visualisation
with open("../output/goal_distribution_raw.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["bucket_idx", "goal_count", "goal_log_count"])
    for i in range(max(len(goal_bins), len(log_bins))):
        g = goal_bins[i]["n"] if i < len(goal_bins) else 0
        l = log_bins[i]["n"]  if i < len(log_bins)  else 0
        w.writerow([i, g, l])

# ASCII quick-look (top-10 buckets)
print("=== goal (raw) — top 10 buckets by count ===")
for r in sorted(goal_bins, key=lambda x: -x["n"])[:10]:
    bar = "█" * min(40, r["n"] // 1000)
    print(f"  bucket {int(r['goal_bucket']):3d}  {r['n']:>7,}  {bar}")

print("\n=== goal_log — top 10 buckets by count ===")
for r in sorted(log_bins, key=lambda x: -x["n"])[:10]:
    bar = "█" * min(40, r["n"] // 1000)
    print(f"  bucket {int(r['goal_log_bucket']):3d}  {r['n']:>7,}  {bar}")

print("\nHistogram data saved to ../output/goal_distribution_raw.csv")


=== goal (raw) — top 10 buckets by count ===
  bucket   0  193,105  ████████████████████████████████████████
  bucket   1    5,222  █████
  bucket   3    1,223  █
  bucket   2    1,175  █
  bucket   6      513  
  bucket   4      406  
  bucket  12      301  
  bucket  60      262  
  bucket   5      148  
  bucket   9      143  

=== goal_log — top 10 buckets by count ===
  bucket  31   22,269  ██████████████████████
  bucket  34   16,411  ████████████████
  bucket  30   14,173  ██████████████
  bucket  25   12,881  ████████████
  bucket  37   12,607  ████████████
  bucket  23   10,874  ██████████
  bucket  28   10,871  ██████████
  bucket  33   10,288  ██████████
  bucket  36    9,849  █████████
  bucket  29    8,193  ████████

Histogram data saved to ../output/goal_distribution_raw.csv


## 3. Duration Feature
Campaign duration (deadline − launched_at) is a strong predictor: campaigns that run too long or too short tend to fail.

In [12]:
df_log = df_log.withColumn("duration", col("deadline") - col("launched_at"))
df_log = df_log.withColumn("staff_pick", col("staff_pick").cast("integer"))

print("Duration stats:")
df_log.select("duration").describe().show()

# Check for any negative durations (data quality)
neg_dur = df_log.filter(col("duration") < 0).count()
print(f"Negative durations (invalid): {neg_dur}")

Duration stats:
+-------+------------------+
|summary|          duration|
+-------+------------------+
|  count|            203341|
|   mean|2857372.3807299067|
| stddev|1057557.4274196366|
|    min|             86400|
|    max|          10418949|
+-------+------------------+

Negative durations (invalid): 0


## 4. Error Handling and Data Lineage

In [13]:
# Error handling: drop rows with nulls introduced during feature engineering
before_drop = df_log.count()
df_clean = df_log.dropna(subset=["goal_log", "duration", "staff_pick", "category", "country", "state"])
after_drop = df_clean.count()

print(f"Rows before dropna: {before_drop}")
print(f"Rows after  dropna: {after_drop}  (removed {before_drop - after_drop})")

# Data lineage summary
print("\n--- Data Lineage ---")
print("Source         : CSV kickstarter_2022-2021_unique_blurbs.csv")
print("Ingestion      : data_engineering.py → Parquet (partitioned by country)")
print("Transformations: log1p(goal) → goal_log")
print("               : deadline - launched_at → duration")
print("               : staff_pick cast to integer")
print("               : StringIndexer on category, country")
print("               : VectorAssembler → features vector")

Rows before dropna: 203341
Rows after  dropna: 203341  (removed 0)

--- Data Lineage ---
Source         : CSV kickstarter_2022-2021_unique_blurbs.csv
Ingestion      : data_engineering.py → Parquet (partitioned by country)
Transformations: log1p(goal) → goal_log
               : deadline - launched_at → duration
               : staff_pick cast to integer
               : StringIndexer on category, country
               : VectorAssembler → features vector


## 5. String Indexing and Vector Assembly

In [14]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

cat_indexer = StringIndexer(
    inputCol="category", outputCol="category_index", handleInvalid="keep"
)
ctr_indexer = StringIndexer(
    inputCol="country",  outputCol="country_index",  handleInvalid="keep"
)
assembler = VectorAssembler(
    inputCols=["goal_log", "duration", "staff_pick", "category_index", "country_index"],
    outputCol="features"
)

# Build a feature pipeline (no ML estimator yet)
feat_pipeline = Pipeline(stages=[cat_indexer, ctr_indexer, assembler])
feat_model    = feat_pipeline.fit(df_clean)
df_features   = feat_model.transform(df_clean)

# Show feature vectors
df_features.select("features", "state").show(5, truncate=False)

+------------------------------------------+-----+
|features                                  |state|
+------------------------------------------+-----+
|[8.517393171418904,3348060.0,0.0,32.0,0.0]|0    |
|[7.601402334583733,1728000.0,0.0,4.0,0.0] |1    |
|[6.111467339502679,1604508.0,1.0,61.0,0.0]|1    |
|[7.523481312573497,2592000.0,0.0,60.0,0.0]|1    |
|[8.853808274977197,5184000.0,0.0,61.0,0.0]|0    |
+------------------------------------------+-----+
only showing top 5 rows



## 7. Caching Strategy

In [15]:
from pyspark import StorageLevel

# Repartition to 200 for balanced shuffle in ML algorithms
df_clean = df_clean.repartition(200)

# Cache with MEMORY_AND_DISK: spills to disk if RAM fills
# Justification: df_clean is used by train/test split AND feature fitting
# Caching avoids re-reading Parquet twice during Pipeline.fit()
df_clean.persist(StorageLevel.MEMORY_AND_DISK)
cached_count = df_clean.count()

print(f"Cached rows: {cached_count}")
print("Partitions :", df_clean.rdd.getNumPartitions())

# Export for next notebook
df_clean.write.mode("overwrite").parquet("../data/parquet/kickstarter_features")
print("Feature-engineered dataset written to parquet.")

df_clean.unpersist()
spark.stop()

Cached rows: 203341
Partitions : 200
Feature-engineered dataset written to parquet.
